We keep the DummyTransformerBlock and DummyLayerNorm blank, and pass the input to the model and receives the logits as output

![image_1769518945790.png](./image_1769518945790.png "image_1769518945790.png")

![image_1769518868260.png](./image_1769518868260.png "image_1769518868260.png")

![image_1769519177913.png](./image_1769519177913.png "image_1769519177913.png")

![image_1769518888596.png](./image_1769518888596.png "image_1769518888596.png")

![image_1769518920577.png](./image_1769518920577.png "image_1769518920577.png")

In [0]:
GPT_CONFIG_124M = {
    "vocab_size": 50257,    # Vocabulary size
    "context_length": 1024, # Context length
    "emb_dim": 768,         # Embedding dimension
    "n_heads": 12,          # Number of attention heads
    "n_layers": 12,         # Number of layers
    "drop_rate": 0.1,       # Dropout rate
    "qkv_bias": False       # Query-Key-Value bias
}

In [0]:
%pip install torch

In [0]:
%pip install tiktoken

In [0]:
import torch
import torch.nn as nn

NOTE: The forward method in your DummyGPTModel class will be called when you pass input data to an instance of the model using the call method, which is inherited from nn.Module. For example, if you have an instance model and a tensor in_idx, you would call model(in_idx). This automatically invokes the forward method and returns the output logits. This is standard PyTorch behavior

In [0]:
class DummyGPTModel(nn.Module):
    def __init__(self, cfg):
        super().__init__()
        print(cfg)
        #{'vocab_size': 50257, 'context_length': 1024, 'emb_dim': 768, 'n_heads': 12, 'n_layers': 12, 'drop_rate': 0.1, 'qkv_bias': False}
        self.tok_emb = nn.Embedding(cfg["vocab_size"], cfg["emb_dim"]) #50257, 768
        self.pos_emb = nn.Embedding(cfg["context_length"], cfg["emb_dim"])#1024,768 
        self.drop_emb = nn.Dropout(cfg["drop_rate"])
        
        # Use a placeholder for TransformerBlock
        self.trf_blocks = nn.Sequential(
            *[DummyTransformerBlock(cfg) for _ in range(cfg["n_layers"])])
        
        # Use a placeholder for LayerNorm
        self.final_norm = DummyLayerNorm(cfg["emb_dim"])
        self.out_head = nn.Linear(
            cfg["emb_dim"], cfg["vocab_size"], bias=False
        )

    def forward(self, in_idx):
        batch_size, seq_len = in_idx.shape
        print(batch_size, seq_len) #2,4
        tok_embeds = self.tok_emb(in_idx)
        print("token embedding",tok_embeds)
        pos_embeds = self.pos_emb(torch.arange(seq_len, device=in_idx.device))
        print("position embedding",pos_embeds)
        x = tok_embeds + pos_embeds
        x = self.drop_emb(x)
        x = self.trf_blocks(x)
        x = self.final_norm(x)
        logits = self.out_head(x)
        return logits

In [0]:
class DummyTransformerBlock(nn.Module):
    def __init__(self, cfg):
        super().__init__()
        # A simple placeholder

    def forward(self, x):
        # This block does nothing and just returns its input.
        return x

In [0]:
class DummyLayerNorm(nn.Module):
    def __init__(self, normalized_shape, eps=1e-5):
        super().__init__()
        # The parameters here are just to mimic the LayerNorm interface.

    def forward(self, x):
        # This layer does nothing and just returns its input.
        return x

In [0]:
class LayerNorm(nn.Module):
    def __init__(self, emb_dim):
        super().__init__()
        self.eps = 1e-5
        self.scale = nn.Parameter(torch.ones(emb_dim))
        self.shift = nn.Parameter(torch.zeros(emb_dim))

    def forward(self, x):
        mean = x.mean(dim=-1, keepdim=True)
        var = x.var(dim=-1, keepdim=True, unbiased=False)
        norm_x = (x - mean) / torch.sqrt(var + self.eps)
        return self.scale * norm_x + self.shift

In [0]:
import tiktoken

tokenizer = tiktoken.get_encoding("gpt2")

batch = []

txt1 = "Every effort moves you"
txt2 = "Every day holds a"

batch.append(torch.tensor(tokenizer.encode(txt1)))
batch.append(torch.tensor(tokenizer.encode(txt2)))
batch = torch.stack(batch, dim=0)
print(batch)

In [0]:
GPT_CONFIG_124M = {
    "vocab_size": 50257,    # Vocabulary size
    "context_length": 1024, # Context length
    "emb_dim": 768,         # Embedding dimension
    "n_heads": 12,          # Number of attention heads
    "n_layers": 12,         # Number of layers
    "drop_rate": 0.1,       # Dropout rate
    "qkv_bias": False       # Query-Key-Value bias
}

In [0]:
torch.manual_seed(123)
model = DummyGPTModel(GPT_CONFIG_124M)

logits = model(batch)
print("Output shape:", logits.shape)
print(logits)

# Layer Normalization in the LLM Architecture

![image_1769526526306.png](./image_1769526526306.png "image_1769526526306.png")

![image_1769526551483.png](./image_1769526551483.png "image_1769526551483.png")

In [0]:
torch.manual_seed(123)

# create 2 training examples with 5 dimensions (features) each
batch_example = torch.randn(2, 5) 

layer = nn.Sequential(nn.Linear(5, 6), nn.ReLU())
out = layer(batch_example)
print(out)

![image_1769526504743.png](./image_1769526504743.png "image_1769526504743.png")

In [0]:
mean = out.mean(dim=-1, keepdim=True)
var = out.var(dim=-1, keepdim=True)

print("Mean:\n", mean)
print("Variance:\n", var)

In [0]:
out_norm = (out - mean) / torch.sqrt(var)
print("Normalized layer outputs:\n", out_norm)

mean = out_norm.mean(dim=-1, keepdim=True)
var = out_norm.var(dim=-1, keepdim=True)
print("Mean:\n", mean)
print("Variance:\n", var)

In [0]:
torch.set_printoptions(sci_mode=False)
print("Mean:\n", mean)
print("Variance:\n", var)

In [0]:
ln = LayerNorm(emb_dim=5)
out_ln = ln(batch_example)

In [0]:
mean = out_ln.mean(dim=-1, keepdim=True)
var = out_ln.var(dim=-1, unbiased=False, keepdim=True)

print("Mean:\n", mean)
print("Variance:\n", var)

# Implementing a feed forward network with GELU activations

Why we need feed forward network with GELU activations 

It gets right to the heart of why modern LLMs are so powerful.

The feed-forward network (FFN) with GELU activations is a critical component within each Transformer block, which is the fundamental building block of LLMs like GPT.

---

**Here's why we need it:**

- **Adding Non-linearity:**  
  The FFN's primary role is to introduce non-linearity into the model. Without activation functions like GELU, the entire neural network would just be a series of linear transformations, no matter how many layers you stack. This would severely limit its ability to learn complex patterns and relationships in language. GELU allows the network to model intricate, non-linear functions.

- **Processing Attention Outputs:**  
  After the Multi-Head Attention mechanism processes the input sequence, the FFN takes that output and further transforms it. It essentially allows the model to "think deeper" about the information gathered by the attention layers, refining the representations.

---

**GELU's Specific Advantages:**

- **Smoothness:**  
  Unlike ReLU, GELU is smooth and differentiable everywhere. This is beneficial for gradient-based optimization during training, leading to more stable and efficient learning.

- **Avoiding "Dead Neurons":**  
  The video explained how ReLU can suffer from the "dead neuron" problem where neurons stop learning if their input is negative. GELU, by having a small, non-zero output for negative inputs, helps prevent this, ensuring more neurons remain active and contribute to learning.

- **Improved Performance:**  
  Empirically, GELU has been shown to outperform ReLU in many Transformer-based models, including its use in GPT-2, contributing to better overall model performance and robustness.

---

So, in essence, the FFN with GELU activations provides the necessary non-linearity and computational depth for the Transformer to learn the incredibly complex patterns required to understand and generate human language effectively.

In [0]:
class GELU(nn.Module):
    def __init__(self):
        super().__init__()

    def forward(self, x):
        return 0.5 * x * (1 + torch.tanh(
            torch.sqrt(torch.tensor(2.0 / torch.pi)) * 
            (x + 0.044715 * torch.pow(x, 3))
        ))

In [0]:
import matplotlib.pyplot as plt

gelu, relu = GELU(), nn.ReLU()

# Some sample data
x = torch.linspace(-3, 3, 100)
y_gelu, y_relu = gelu(x), relu(x)

plt.figure(figsize=(8, 3))
for i, (y, label) in enumerate(zip([y_gelu, y_relu], ["GELU", "ReLU"]), 1):
    plt.subplot(1, 2, i)
    plt.plot(x, y)
    plt.title(f"{label} activation function")
    plt.xlabel("x")
    plt.ylabel(f"{label}(x)")
    plt.grid(True)

plt.tight_layout()
plt.show()

In [0]:
class FeedForward(nn.Module):
    def __init__(self, cfg):
        super().__init__()
        self.layers = nn.Sequential(
            nn.Linear(cfg["emb_dim"], 4 * cfg["emb_dim"]),
            GELU(),
            nn.Linear(4 * cfg["emb_dim"], cfg["emb_dim"]),
        )

    def forward(self, x):
        return self.layers(x)

In [0]:
print(GPT_CONFIG_124M["emb_dim"])

In [0]:
ffn = FeedForward(GPT_CONFIG_124M)

# input shape: [batch_size, num_token, emb_size]
x = torch.rand(2, 3, 768) 
out = ffn(x)
print(out.shape)

![image_1769795737177.png](./image_1769795737177.png "image_1769795737177.png")

![image_1769795758615.png](./image_1769795758615.png "image_1769795758615.png")

![image_1769795774431.png](./image_1769795774431.png "image_1769795774431.png")

# Adding shortcut connections or Shortcut connections

# Why We Need Shortcut (Skip) Connections in GPT Architecture

Shortcut connections, also known as skip or residual connections, are crucial for the GPT (Generative Pre-trained Transformer) architecture. They directly address the **vanishing gradient problem**, which occurs when gradients become progressively smaller during backpropagation in deep neural networks, leading to stagnant learning and delayed convergence.

---

## Key Benefits of Shortcut Connections

- **Prevent Vanishing Gradients:**  
  Shortcut connections create alternative paths for gradients to flow by skipping one or more layers. This ensures gradients remain significant even in the earlier layers of a deep network. Mathematically, the addition of a "+1" term in the gradient calculation prevents it from approaching zero, eliminating the vanishing gradient problem.

- **Enable Deeper Networks:**  
  By ensuring consistent gradient flow, shortcut connections make it possible to train very deep neural networks, like the many layers found in GPT models, which would otherwise be difficult or impossible to optimize effectively.

- **Preserve Information:**  
  They add the output of one layer to the output of a later layer, allowing the network to retain original input information while learning transformations. This helps the model maintain context across multiple layers.

- **Smooth Loss Landscape:**  
  Neural networks without skip connections have complex loss landscapes with many local minima. Skip connections result in a much smoother landscape, facilitating more stable and efficient training.

- **Stabilize Training and Improve Performance:**  
  By facilitating more effective training and ensuring consistent gradient flow across layers, shortcut connections lead to better overall model performance and more stable training of GPT models.

---

## How Shortcut Connections Are Used in Transformers

In the Transformer block, shortcut connections are applied by **adding the input of a sub-layer (like multi-head attention or feed-forward network) to its output before layer normalization**. This mechanism ensures that information and gradients can flow directly through the network, making deep Transformer architectures feasible.

---

The video demonstrates the practical impact of shortcut connections through a Python implementation, showing that **without them, gradient values significantly decrease in earlier layers**, whereas **with shortcut connections, the gradient flow remains stable and much higher**.

![image_1769851373706.png](./image_1769851373706.png "image_1769851373706.png")

Example of Neural network

In [0]:
import torch
import torch.nn as nn

In [0]:
class ExampleDeepNeuralNetwork(nn.Module):
    def __init__(self, layer_sizes, use_shortcut):
        super().__init__()
        self.use_shortcut = use_shortcut
        self.layers = nn.ModuleList([
            nn.Sequential(nn.Linear(layer_sizes[0], layer_sizes[1]), GELU()),
            nn.Sequential(nn.Linear(layer_sizes[1], layer_sizes[2]), GELU()),
            nn.Sequential(nn.Linear(layer_sizes[2], layer_sizes[3]), GELU()),
            nn.Sequential(nn.Linear(layer_sizes[3], layer_sizes[4]), GELU()),
            nn.Sequential(nn.Linear(layer_sizes[4], layer_sizes[5]), GELU())
        ])

    def forward(self, x):
        for layer in self.layers:
            # Compute the output of the current layer
            print("Layer-->",layer(x))
            layer_output = layer(x)
            # Check if shortcut can be applied
            if self.use_shortcut and x.shape == layer_output.shape:
                x = x + layer_output
            else:
                x = layer_output
        return x

In [0]:
def print_gradients(model, x):
    # Forward pass
    output = model(x)
    target = torch.tensor([[0.]])

    # Calculate loss based on how close the target
    # and output are
    loss = nn.MSELoss()
    loss = loss(output, target)
    
    # Backward pass to calculate the gradients
    loss.backward()

    for name, param in model.named_parameters():
        if 'weight' in name:
            # Print the mean absolute gradient of the weights
            print(f"{name} has gradient mean of {param.grad.abs().mean().item()}")

In [0]:
layer_sizes = [3, 3, 3, 3, 3, 1]  

sample_input = torch.tensor([[1., 0., -1.]])

torch.manual_seed(123)
model_without_shortcut = ExampleDeepNeuralNetwork(
    layer_sizes, use_shortcut=False
)
print_gradients(model_without_shortcut, sample_input)

In [0]:
torch.manual_seed(123)
model_with_shortcut = ExampleDeepNeuralNetwork(
    layer_sizes, use_shortcut=True
)
print_gradients(model_with_shortcut, sample_input)